# Stroke detection

## Load and inspect data
Load pickle file and inspect contents

In [ ]:
# Import necessary pyologger utilities
import os

from pyologger.utils.folder_manager import *
from pyologger.utils.event_manager import *
from pyologger.plot_data.plotter import *
from pyologger.io_operations.base_exporter import *
from pyologger.utils.data_manager import *
from pyologger.process_data.peak_detect import *
from pyologger.process_data.odba import *


def _resolve_channel_alias(requested_channel, available_channels):
    """Allow x/y/z <-> ax/ay/az aliases for accel channels."""
    if requested_channel in available_channels:
        return requested_channel
    alias_map = {
        "ax": "x", "ay": "y", "az": "z",
        "x": "ax", "y": "ay", "z": "az",
        "gx": "x", "gy": "y", "gz": "z",
    }
    alias = alias_map.get(str(requested_channel).lower())
    if alias in available_channels:
        return alias
    return requested_channel


# dataset_id = "pale-adult-lion_vid-accel_africa_TW"
# deployment_id = "2015-06-16_pale-010"

# dataset_id = "mile-adult-sese_vdr_argentina_RD-KM"
# deployment_id = "2013-11-09_mile-008"

# dataset_id = "oror-adult-orca_hr-sr-vid_sw_JKB-PP"
# deployment_id = "2023-10-26_oror-001"

# dataset_id = "caca-chmy_juv-adult_CW-PP-KS"
# deployment_id = "2015-09-06_caca-001"

# dataset_id = "nesc-adult-hi-monk-seal_dive-imu_SR-MB"
# deployment_id = "2019-02-05_nesc-007"

dataset_id = "mian-juv-nese_sleep_lml-ano_JKB"
deployment_id = "2019-10-25_mian-001"
# deployment_id = "2021-04-17_mian-011"

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()
# Streamlit load data
animal_id, dataset_id, deployment_id, dataset_folder, deployment_folder, data_pkl, param_manager = select_and_load_deployment(
    data_dir,
    dataset_id=dataset_id,
    deployment_id=deployment_id,
)
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')



In [ ]:
current_processing_step = "Processing Step 04 IN PROGRESS."
param_manager.add_to_config("current_processing_step", current_processing_step)

In [ ]:
# For stroke workflow we can operate with corrected/dynamic/calibrated accelerometer signals.
critical_signal_candidates = ['dynamic_accel', 'corrected_acc', 'calibrated_acc', 'accelerometer']
critical_signal = next(
    (
        sig
        for sig in critical_signal_candidates
        if sig in data_pkl.signal_data and data_pkl.signal_data[sig] is not None
    ),
    None,
)

# Check if stroke_rate exists; if so, ensure units are spm
converted = False
if 'stroke_rate' in data_pkl.signal_data and data_pkl.signal_data['stroke_rate'] is not None:
    sr_df = data_pkl.signal_data['stroke_rate']
    sr_info = data_pkl.signal_info.get('stroke_rate', {})
    metadata = sr_info.get('metadata', {})
    channels = sr_info.get('channels', [c for c in sr_df.columns if c != 'datetime'])

    hz_aliases = {'hz', 'hertz', '1/s', '1/sec', 'sec^-1', 's^-1', 'per second'}
    spm_aliases = {'spm', 'spm', 'spm', 'spm', '1/min', 'min^-1', 'per minute'}

    for ch in channels:
        if ch not in sr_df.columns:
            continue
        unit = str(metadata.get(ch, {}).get('unit', '')).lower()
        is_hz = (unit in hz_aliases) or ('hz' in unit)
        is_spm = (unit in spm_aliases) or ('spm' in unit)

        if is_hz and not is_spm:
            sr_df[ch] = sr_df[ch] * 60.0
            metadata.setdefault(ch, {})
            metadata[ch]['unit'] = 'spm'
            converted = True

    if converted:
        sr_info['metadata'] = metadata
        sr_info['transformation_log'] = sr_info.get('transformation_log', [])
        sr_info['transformation_log'].append('converted_stroke_rate_hz_to_spm')
        data_pkl.signal_data['stroke_rate'] = sr_df
        data_pkl.signal_info['stroke_rate'] = sr_info
        print("Converted stroke_rate from Hz to spm.")

# If stroke_rate was converted above, persist the change immediately
if converted:
    print("✅ stroke_rate converted to spm. Saving updated data.")
    print(current_processing_step)
    param_manager.add_to_config("current_processing_step", current_processing_step)
    with open(pkl_path, 'wb') as file:
        pickle.dump(data_pkl, file)
    print("Pickle file updated.")

# Only proceed with critical signal check if stroke_rate does not already exist
if 'stroke_rate' in data_pkl.signal_data and data_pkl.signal_data['stroke_rate'] is not None:
    skip_step = True
    print("✅ stroke_rate already exists. Skipping processing.")
else:
    # If critical signal doesn't exist, create flag to skip step.
    if critical_signal is None:
        print(f"⚠️ None of the required signals were found: {critical_signal_candidates}. Skipping processing.")
        skip_step = True
        print(f'‼️ DO NOT PROCEED - Skip_step: {skip_step} due to missing required signals: {critical_signal_candidates}')
    else:
        # signals exists and can be processed normally
        skip_step = False
        print(f'✅ Proceed - Skip_step: {skip_step}. Using accelerometer source signal: {critical_signal}.')



## Retrieve relevant configuration settings

In [ ]:
# Retrieve values from config
variables = [
    "calm_horizontal_start_time",
    "calm_horizontal_end_time",
    "zoom_window_start_time",
    "zoom_window_end_time",
    "overlap_start_time",
    "overlap_end_time",
]
settings = param_manager.get_from_config(variables, section="settings")

# Assign retrieved values to variables
CALM_HORIZONTAL_START_TIME = settings.get("calm_horizontal_start_time")
CALM_HORIZONTAL_END_TIME = settings.get("calm_horizontal_end_time")
ZOOM_WINDOW_START_TIME = settings.get("zoom_window_start_time")
ZOOM_WINDOW_END_TIME = settings.get("zoom_window_end_time")
OVERLAP_START_TIME = settings.get("overlap_start_time")
OVERLAP_END_TIME = settings.get("overlap_end_time")

if None in {OVERLAP_START_TIME, OVERLAP_END_TIME, ZOOM_WINDOW_START_TIME, ZOOM_WINDOW_END_TIME}:
    raise ValueError("One or more required time values were not found in the config file.")



In [ ]:
data_pkl.signal_data['corrected_acc']['datetime'].min()

In [ ]:
data_pkl.signal_data['corrected_acc']['datetime'].max()

In [ ]:
data_pkl.signal_info['accelerometer']

## Find time chunk when stroking is dominant activity
Use interactive plot to locate a start time and end time when stroking is the dominant activity.

In [ ]:
TARGET_SAMPLING_RATE = 10

notes_to_plot = {
    'heartbeat_manual_ok': {'signal': 'ecg', 'symbol': 'triangle-down', 'color': 'blue'},
    'heartbeat_auto_detect_accepted': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'green'},
    'heartbeat_auto_detect_rejected': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'red'}
}

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=['ecg', 'depth', 'corrected_acc', 'corrected_gyr', 'prh'],
    time_range=(OVERLAP_START_TIME, OVERLAP_END_TIME),
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_start_time=ZOOM_WINDOW_START_TIME,
    zoom_end_time=ZOOM_WINDOW_END_TIME,
    zoom_range_selector_channel='depth',
    plot_event_values=[],
)
# fig.show()
fig.show_dash(mode="inline")



In [ ]:
# Retrieve timezone from deployment info
timezone = data_pkl.deployment_info['Time Zone']

# Define placeholder timestamps for calm period in the retrieved timezone
stroking_start_time = OVERLAP_START_TIME
stroking_end_time = OVERLAP_END_TIME

# Use ParamManager to add both stroking start and end times to the config in the desired section
param_manager.add_to_config(
    entries={
        "stroking_start_time": str(stroking_start_time),
        "stroking_end_time": str(stroking_end_time)
    },
    section="settings"
)



In [ ]:
# CHANGE AS NEEDED

detection_mode = "stroke_rate"
overwrite = False



In [ ]:
# Define parent signal options
parent_signal_options = list(data_pkl.signal_data.keys())
if detection_mode == "heart_rate":
    default_parent_signal = "ecg"
else:
    stroke_parent_from_config = param_manager.get_from_config(
        variable_names=["STROKE_PARENT_SIGNAL"],
        section="stroke_peak_detection_settings"
    ).get("STROKE_PARENT_SIGNAL")
    # Prefer gyro for stroke detection, but allow corrected/calibrated acc-only datasets.
    stroke_parent_candidates = ["dynamic_accel", "corrected_gyr", "gyroscope", "corrected_acc", "calibrated_acc", "accelerometer"]
    if stroke_parent_from_config in data_pkl.signal_data:
        default_parent_signal = stroke_parent_from_config
    else:
        default_parent_signal = next((sig for sig in stroke_parent_candidates if sig in data_pkl.signal_data), "corrected_gyr")

animal_id = data_pkl.animal_info['Animal_ID']
print(f"Detected animal ID: {animal_id}")

# User input for parent signal
if overwrite:
    print(f"Available parent signals: {parent_signal_options}")
    parent_signal = input(f"Choose parent signal (default: {default_parent_signal}): ").strip()
    if not parent_signal or parent_signal not in parent_signal_options:
        parent_signal = default_parent_signal
else:
    parent_signal = default_parent_signal

# Get available channels
if parent_signal in data_pkl.signal_data:
    available_channels = [c for c in data_pkl.signal_data[parent_signal].columns if c != 'datetime']
elif parent_signal in data_pkl.signal_data:
    available_channels = [c for c in data_pkl.signal_data[parent_signal].columns if c != 'datetime']
else:
    available_channels = []

# Default channel logic
if detection_mode == "heart_rate":
    default_channel = "ecg"
elif detection_mode == "stroke_rate":
    stroke_channel_from_config = param_manager.get_from_config(
        variable_names=["STROKE_CHANNEL"],
        section="stroke_peak_detection_settings"
    ).get("STROKE_CHANNEL")

    if stroke_channel_from_config:
        default_channel = _resolve_channel_alias(stroke_channel_from_config, available_channels)
        if default_channel in available_channels:
            pass
        else:
            default_channel = None
    else:
        default_channel = None

    if default_channel is None:
        if parent_signal in ("corrected_acc", "calibrated_acc", "accelerometer", "dynamic_accel"):
            if animal_id.startswith(('nesc', 'mian')):
                preferred_channels = ['ax', 'x', 'ay', 'y', 'az', 'z']
            elif animal_id.startswith(('oror', 'bamu')):
                preferred_channels = ['ay', 'y', 'ax', 'x', 'az', 'z']
            else:
                preferred_channels = ['ax', 'x', 'ay', 'y', 'az', 'z']
        else:
            if animal_id.startswith(('nesc', 'mian')):
                preferred_channels = ['gx', 'gy', 'gz']
            elif animal_id.startswith(('oror', 'bamu')):
                preferred_channels = ['gy', 'gx', 'gz']
            else:
                preferred_channels = ['gy', 'gx', 'gz']
        default_channel = next((ch for ch in preferred_channels if ch in available_channels), None)
        if default_channel is None and available_channels:
            default_channel = available_channels[0]
else:
    default_channel = available_channels[0] if available_channels else None

# User input for channel
if overwrite:
    print(f"Available channels: {available_channels}")
    channel = input(f"Choose channel (default: {default_channel}): ").strip()
    if not channel or channel not in available_channels:
        channel = default_channel
else:
    channel = default_channel

# Configure signals
signal_df = data_pkl.signal_data[parent_signal]
if channel is None or channel not in signal_df.columns:
    raise ValueError(f"Channel '{channel}' not found in parent signal '{parent_signal}'. Available channels: {available_channels}")
signal = signal_df[channel]
datetime_signal = signal_df['datetime']
sampling_rate = calculate_sampling_frequency(datetime_signal.head())

# Define the default time range based on the signal's datetime column
signal_start = datetime_signal.min()
signal_end = datetime_signal.max()

# Determine time range based on user input if overwrite is True
if overwrite:
    print(f"Signal time range: {signal_start} to {signal_end}")
    start_time_input = input(f"Enter start time (default: {signal_start}): ").strip()
    end_time_input = input(f"Enter end time (default: {signal_end}): ").strip()
    start_datetime = pd.Timestamp(start_time_input) if start_time_input else signal_start
    end_datetime = pd.Timestamp(end_time_input) if end_time_input else signal_end
else:
    start_datetime = signal_start
    end_datetime = signal_end

# Filter signal based on the selected time range
time_mask = (datetime_signal >= start_datetime) & (datetime_signal <= end_datetime)
signal_subset = signal[time_mask]
datetime_subset = datetime_signal[time_mask]
signal_subset_df = signal_df[
    (signal_df['datetime'] >= start_datetime)
    & (signal_df['datetime'] <= end_datetime)
]

# Output the results
print(f"Time range selected: {start_datetime} to {end_datetime}")
print(f"Signal subset size: {len(signal_subset)}")

# Retrieve parameters for peak detection
params = param_manager.get_from_config(
    variable_names=[
        "BROAD_LOW_CUTOFF", "BROAD_HIGH_CUTOFF", "NARROW_LOW_CUTOFF", "NARROW_HIGH_CUTOFF",
        "FILTER_ORDER", "SPIKE_THRESHOLD", "SMOOTH_SEC_MULTIPLIER", "WINDOW_SIZE_MULTIPLIER",
        "NORMALIZATION_NOISE", "PEAK_HEIGHT", "PEAK_DISTANCE_SEC", "SEARCH_RADIUS_SEC",
        "MIN_PEAK_HEIGHT", "MAX_PEAK_HEIGHT", "enable_bandpass", "enable_spike_removal",
        "enable_absolute", "enable_smoothing", "enable_normalization", "enable_refinement",
        "ANTI_DOUBLE_GAP_FACTOR", "HR_CONFLICT_RR_FACTOR", "PICK_LAST_IN_CONFLICT_PAIR"
    ],
    section="hr_peak_detection_settings" if detection_mode == "Heart Rate" else "stroke_peak_detection_settings"
)



In [ ]:
overwrite = False  # If needed, change to true and rewrite settings here

# Add parameters to the config file (if not already present)
if overwrite | any(value is None for value in params.values()):
    # Define default parameters for peak detection with simple structure (no descriptions here)
    params = {
        "BROAD_LOW_CUTOFF": 0.05,  # Hz, lower cutoff for the broad bandpass filter
        "BROAD_HIGH_CUTOFF": 10,  # Hz, upper cutoff for the broad bandpass filter
        "NARROW_LOW_CUTOFF": 0.1,  # Hz, lower cutoff for the narrow bandpass filter
        "NARROW_HIGH_CUTOFF": 2.0,  # Hz, upper cutoff for the narrow bandpass filter
        "FILTER_ORDER": 2,  # Order of the bandpass filter, affects sharpness
        "SPIKE_THRESHOLD": 400,  # Threshold for removing large spikes (e.g., noise or artifacts)
        "SMOOTH_SEC_MULTIPLIER": 0.41,  # Multiplier for calculating the smoothing window size (3 for HR)
        "WINDOW_SIZE_MULTIPLIER": 15.5,  # Multiplier for calculating sliding window size (if this is too big it will lump all strokes into a plateau)
        "NORMALIZATION_NOISE": 1e-10,  # Small constant to avoid division by zero in normalization
        "PEAK_HEIGHT": -0.9,  # Minimum amplitude (height) for peak detection
        "PEAK_DISTANCE_SEC": 0.5,  # Minimum time between detected peaks (in seconds)
        "SEARCH_RADIUS_SEC": 0.3,  # Time range for refining the peak location (in seconds)
        "MIN_PEAK_HEIGHT": 150,  # Minimum acceptable amplitude for detected peaks; original units
        "MAX_PEAK_HEIGHT": 1000000,  # Maximum acceptable amplitude for detected peaks; original units
        "enable_bandpass": True,  # Enable/disable bandpass filtering
        "enable_spike_removal": False,  # Enable/disable spike removal
        "enable_absolute": False,  # Enable/disable abs() transformation of signal (only use if HR, not for stroke rate)
        "enable_smoothing": True,  # Enable/disable smoothing
        "enable_normalization": True,  # Enable/disable sliding window normalization
        "enable_refinement": True,  # Enable/disable peak refinement
        "ANTI_DOUBLE_GAP_FACTOR": 0.75,  # Generalized anti-double detection factor (cross-workflow consistency)
        "HR_CONFLICT_RR_FACTOR": 0.75,  # Legacy alias for backward compatibility
        "PICK_LAST_IN_CONFLICT_PAIR": True,  # keep later peak in conflict pairs by default
    }
    param_manager.add_to_config(entries=params, section="stroke_peak_detection_settings")
else:
    print("Settings loaded from config file, not overwritten.")



In [ ]:
sampling_rate

In [ ]:
params['BROAD_HIGH_CUTOFF']

In [ ]:
# Run peak detection
results = peak_detect(
    signal=signal_subset,
    sampling_rate=sampling_rate,
    datetime_series=datetime_subset,
    broad_lowcut=params["BROAD_LOW_CUTOFF"],
    broad_highcut=params["BROAD_HIGH_CUTOFF"],
    narrow_lowcut=params["NARROW_LOW_CUTOFF"],
    narrow_highcut=params["NARROW_HIGH_CUTOFF"],
    filter_order=params["FILTER_ORDER"],
    spike_threshold=params["SPIKE_THRESHOLD"],
    smooth_sec_multiplier=params["SMOOTH_SEC_MULTIPLIER"],
    window_size_multiplier=params["WINDOW_SIZE_MULTIPLIER"],
    normalization_noise=params["NORMALIZATION_NOISE"],
    peak_height=params["PEAK_HEIGHT"],
    peak_distance_sec=params["PEAK_DISTANCE_SEC"],
    search_radius_sec=params["SEARCH_RADIUS_SEC"],
    min_peak_height=params["MIN_PEAK_HEIGHT"],
    max_peak_height=params["MAX_PEAK_HEIGHT"],
    enable_bandpass=params["enable_bandpass"],
    enable_spike_removal=params["enable_spike_removal"],
    enable_absolute=params["enable_absolute"],
    enable_smoothing=params["enable_smoothing"],
    enable_normalization=params["enable_normalization"],
    enable_refinement=params["enable_refinement"]
)

In [ ]:
process_rate(data_pkl, results, signal_subset_df, parent_signal,
             params, sampling_rate, detection_mode)

In [ ]:
results['peak_df']

## Use streamlit app to refine parameters

## Define peak detection parameters for stroke rate detection

## Calculate stroke rate

In [ ]:
TARGET_SAMPLING_RATE = 10

notes_to_plot = {
    'heartbeat_manual_ok': {'signal': 'ecg', 'symbol': 'triangle-down', 'color': 'blue'},
    'heartbeat_auto_detect_accepted': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'green'},
    'heartbeat_auto_detect_rejected': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'red'},
    'strokebeat_auto_detect_accepted': {'signal': 'sr_narrow_bandpass', 'symbol': 'triangle-up', 'color': 'green'},
    'strokebeat_auto_detect_rejected': {'signal': 'sr_narrow_bandpass', 'symbol': 'triangle-up', 'color': 'red'}
}

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=['ecg', 'gyroscope', 'depth', 'corrected_gyr', 'prh', 'stroke_rate', 'sr_broad_bandpass', 'sr_narrow_bandpass', 'sr_smoothed', 'sr_normalized'],
    channels={},
    time_range=(OVERLAP_START_TIME, OVERLAP_END_TIME),
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_start_time=stroking_start_time,
    zoom_end_time=stroking_end_time,
    zoom_range_selector_channel='depth',
    plot_event_values=[],
)

fig.show_dash(mode="inline")



In [ ]:
# Clear the specified keys
keys_to_remove = ['sr_broad_bandpass','sr_narrow_bandpass', 'sr_normalized']
clear_intermediate_signals(data_pkl, remove_keys=keys_to_remove)

initial_event_count = len(data_pkl.event_data)
# Remove events with keys ending in '_rejected'
data_pkl.event_data = data_pkl.event_data[~data_pkl.event_data['key'].str.endswith('_rejected', na=False)]
# Get the final count of events
final_event_count = len(data_pkl.event_data)
# Print the number of removed events
removed_event_count = initial_event_count - final_event_count
print(f"Removed {removed_event_count} events with keys ending in '_rejected'.")


## Get excerpt with stroking only

In [ ]:
# Prefer corrected_acc for ODBA; fall back to calibrated_acc if needed.
odba_source_signal = 'corrected_acc' if 'corrected_acc' in data_pkl.signal_data else 'calibrated_acc'
corrected_acc = data_pkl.signal_data[odba_source_signal]
acc_sampling_rate = calculate_sampling_frequency(corrected_acc['datetime'])
acc_sampling_rate



In [ ]:
stroke_rate_subset = data_pkl.signal_data['stroke_rate'][
    (data_pkl.signal_data['stroke_rate']['datetime'] >= stroking_start_time)
    & (data_pkl.signal_data['stroke_rate']['datetime'] <= stroking_end_time)
]

# Calculate mean stroke rate
mean_stroke_rate = stroke_rate_subset['stroke_rate'].mean()
stroke_hz = mean_stroke_rate / 60
print(f'Stroke rate in Hz: {stroke_hz} Hz.')

fh = stroke_hz / 2
n = 4 * round(acc_sampling_rate / fh)

# Calculate ODBA using VeDBA method
odba_df = compute_odba(corrected_acc, fs=acc_sampling_rate, method='wilson', n=n)

# Print the first few rows of the resulting ODBA DataFrame
print(odba_df.head())

# Optionally store it in signal_data
data_pkl.signal_data['odba'] = odba_df
data_pkl.signal_info['odba'] = {
    "channels": ["odba"],
    "metadata": {
        "odba": {"original_name": "Overall Dynamic Body Acceleration (VeDBA)", "unit": "g"}
    },
    "derived_from_signals": [odba_source_signal],
    "transformation_log": [f"VeDBA calculated with n={n} from {odba_source_signal}"]
}



In [ ]:
TARGET_SAMPLING_RATE = 10

notes_to_plot = {
    'heartbeat_manual_ok': {'signal': 'ecg', 'symbol': 'triangle-down', 'color': 'blue'},
    'heartbeat_auto_detect_accepted': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'green'},
    'heartbeat_auto_detect_rejected': {'signal': 'ecg', 'symbol': 'triangle-up', 'color': 'red'},
    'strokebeat_auto_detect_accepted': {'signal': 'sr_smoothed', 'symbol': 'triangle-up', 'color': 'green'},
    'strokebeat_auto_detect_rejected': {'signal': 'sr_smoothed', 'symbol': 'triangle-up', 'color': 'red'}
}

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=['ecg', 'gyroscope', 'depth', 'corrected_gyr', 'prh', 'stroke_rate', 'sr_smoothed','odba'],
    channels={}, #'corrected_gyr': ['broad_bandpassed_signal']
    time_range=(OVERLAP_START_TIME, OVERLAP_END_TIME),
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_start_time=stroking_start_time,
    zoom_end_time=stroking_end_time,
    zoom_range_selector_channel='depth',
    plot_event_values=[],
)

fig.show()

In [ ]:
current_processing_step = "Processing Step 04. Stroke rate and ODBA calculation complete."
print(current_processing_step)

# Add or update the current_processing_step for the specified deployment
param_manager.add_to_config("current_processing_step", current_processing_step)

# Optional: save new pickle file
with open(pkl_path, 'wb') as file:
        pickle.dump(data_pkl, file)
print("Pickle file updated.")